In [ ]:
import os, json, re, textwrap
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
from pycocotools.coco import COCO


In [ ]:
COCO_ROOT = "/BS/generative_modelling_for_image_understanding/nobackup/data/DETECTRON2_DATASETS/coco"   # <-- change
SPLIT = "val2017"             # or "train2017"
ANN_FILE = os.path.join(COCO_ROOT, "annotations", f"instances_{SPLIT}.json")
IMG_DIR  = os.path.join(COCO_ROOT, SPLIT)

assert os.path.isfile(ANN_FILE), ANN_FILE
assert os.path.isdir(IMG_DIR), IMG_DIR

device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
coco = COCO(ANN_FILE)

cat_ids = coco.getCatIds()
cats = coco.loadCats(cat_ids)
cats_sorted = sorted(cats, key=lambda x: x["id"])

cat_ids_sorted = [c["id"] for c in cats_sorted]
cat_names_sorted = [c["name"] for c in cats_sorted]
catid_to_index = {cid: i for i, cid in enumerate(cat_ids_sorted)}
name_to_idx = {name: i for i, name in enumerate(cat_names_sorted)}

len(cat_names_sorted), cat_names_sorted[:10]


In [ ]:
def get_gt_labels_for_img(img_id: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    present_cids = sorted({a["category_id"] for a in anns})
    present_names = [coco.loadCats([cid])[0]["name"] for cid in present_cids]
    return present_names

# quick check on one image
img_ids = coco.getImgIds()
img_ids[0], get_gt_labels_for_img(img_ids[0])[:100]


In [ ]:
model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"

dtype = torch.float16 if device == "cuda" else torch.float32

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
).to(device).eval()

processor = AutoProcessor.from_pretrained(model_id)

# Important for generation stability
processor.tokenizer.padding_side = "left"

print("Loaded:", model_id, "on", device)


In [ ]:
def show_image(path, title=None, figsize=(5,5)):
    img = Image.open(path).convert("RGB")
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()
    return img

def extract_json_object(text: str):
    # Find first '{' and last '}' and attempt parse (heuristic)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    snippet = text[start:end+1].strip()
    snippet = re.sub(r"^```(json)?\s*", "", snippet)
    snippet = re.sub(r"\s*```$", "", snippet)
    try:
        return json.loads(snippet)
    except Exception:
        return None


In [ ]:
def build_list_prompt(label_list):
    conversation = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": (
                "Identify which COCO-80 classes are present in the image.\n"
                "Only use labels from this list exactly (no synonyms, no extra words).\n"
                "Return ONLY valid JSON:\n"
                "{\"labels\": [\"dog\", \"person\"]}\n"
                "If none: {\"labels\": []}\n\n"
                "Label list:\n" + ", ".join(label_list)
            )},
        ],
    }]
    return processor.apply_chat_template(conversation, add_generation_prompt=True)

@torch.no_grad()
def run_list_mode(image: Image.Image, label_list, max_new_tokens=256):
    prompt = build_list_prompt(label_list)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

    # recommended: make padding explicit for generation
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id,
    )

    # IMPORTANT: decode only the generated completion (not the prompt)
    gen_only = output[0, inputs["input_ids"].shape[1]:]
    text = processor.decode(gen_only, skip_special_tokens=True).strip()

    obj = extract_json_object(text)

    allowed = {x.lower(): x for x in label_list}
    preds = []
    if obj and isinstance(obj.get("labels", None), list):
        for it in obj["labels"]:
            if isinstance(it, str):
                key = it.strip().lower()
                if key in allowed:
                    preds.append(allowed[key])
        preds = sorted(set(preds))

    # No “contains(prompt)” fallback — if JSON fails, return empty and inspect raw_text.
    return {
        "prompt": prompt,
        "raw_text": text,
        "pred_labels": preds,
        "json_ok": obj is not None,
    }


In [ ]:
def build_yesno_prompt(label: str):
    conversation = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": (
                "Question: Is there a '{label}' in this image (COCO sense)?\n"
                "Answer with one character: A (Yes) or B (No).\n"
                "Answer:"
            ).format(label=label)},
        ],
    }]
    return processor.apply_chat_template(conversation, add_generation_prompt=True)

@torch.no_grad()
def score_yesno_teacher_forced(image: Image.Image, labels, chunk_size=16):
    """
    For each label, compute P(A) vs P(B) by scoring the appended token directly.
    This avoids issues where the first generated token is newline/space/etc.
    """
    tok = processor.tokenizer

    # Prefer single-token ids; try both with and without leading space
    def single_token_ids(cands):
        ids = []
        for s in cands:
            t = tok.encode(s, add_special_tokens=False)
            if len(t) == 1:
                ids.append(t[0])
        return list(dict.fromkeys(ids))

    A_ids = single_token_ids(["A", " A"])
    B_ids = single_token_ids(["B", " B"])
    if not A_ids or not B_ids:
        raise RuntimeError("Could not find single-token ids for A/B. Try '1'/'0'.")

    scores = np.zeros((len(labels),), dtype=np.float32)
    prompts_preview = []

    for start in range(0, len(labels), chunk_size):
        chunk = labels[start:start+chunk_size]
        prompts = [build_yesno_prompt(lab) for lab in chunk]
        if start == 0:
            prompts_preview = prompts[:10]

        # Tokenize prompt
        base = processor(images=[image]*len(chunk), text=prompts, return_tensors="pt", padding=True)
        base = {k: v.to(device) for k, v in base.items()}

        # For teacher forcing: we evaluate logprob of A vs B at the next position.
        # We do it by taking logits at last non-pad position and extracting A/B logits.
        out = model(**base)
        attn = base["attention_mask"]
        last_pos = attn.sum(dim=1) - 1
        bidx = torch.arange(out.logits.size(0), device=device)

        # logits for the token *after* last prompt token are in out.logits[bidx, last_pos, :]
        next_logits = out.logits[bidx, last_pos, :]  # (B, V)

        logit_A = torch.logsumexp(next_logits[:, A_ids], dim=1)
        logit_B = torch.logsumexp(next_logits[:, B_ids], dim=1)

        two = torch.stack([logit_A, logit_B], dim=1)
        prob_A = torch.softmax(two, dim=1)[:, 0]  # P(A=Yes)

        scores[start:start+len(chunk)] = prob_A.float().cpu().numpy()

    return scores, prompts_preview


In [ ]:
def run_yesno_mode(image: Image.Image, labels, threshold=0.5, chunk_size=16, topk=10):
    scores, prompt_preview = score_yesno_teacher_forced(image, labels, chunk_size=chunk_size)
    pred = [labels[i] for i in np.where(scores >= threshold)[0].tolist()]
    top_idx = np.argsort(-scores)[:topk].tolist()
    top_scored = [{"label": labels[i], "score": float(scores[i])} for i in top_idx]
    return {
        "prompt_preview_10": "\n\n".join(prompt_preview),
        "threshold": threshold,
        "pred_labels": pred,
        "top_scored": top_scored,
        "scores": scores,
    }


In [ ]:
def choose_hybrid_subset(all_labels, candidates, num_neg=16, seed=123, img_id=0, max_candidates=20):
    cand = list(dict.fromkeys(candidates))  # unique preserve order
    if max_candidates > 0:
        cand = cand[:max_candidates]
    cand_set = set(cand)
    remaining = [l for l in all_labels if l not in cand_set]

    rng = np.random.default_rng(seed + int(img_id))
    k = min(num_neg, len(remaining))
    neg = rng.choice(remaining, size=k, replace=False).tolist() if k > 0 else []
    subset = list(dict.fromkeys(cand + neg))
    return subset

def run_hybrid_mode(
    image: Image.Image,
    img_id: int,
    labels,
    threshold=0.5,
    chunk_size=16,
    max_new_tokens=256,
    num_neg=16,
    seed=123,
    max_candidates=20,
    topk=10,
):
    # 1) list step (strict JSON, decoded from generated tokens only)
    list_res = run_list_mode(image, labels, max_new_tokens=max_new_tokens)
    candidates = list_res["pred_labels"]

    # 2) choose subset to score
    subset = choose_hybrid_subset(
        labels,
        candidates,
        num_neg=num_neg,
        seed=seed,
        img_id=img_id,
        max_candidates=max_candidates,
    )

    # 3) score subset only (use the NEW yes/no scorer)
    subset_scores, subset_prompt_preview = score_yesno_teacher_forced(
        image,
        subset,
        chunk_size=chunk_size,
    )

    # 4) zero-fill full scores
    full_scores = np.zeros((len(labels),), dtype=np.float32)
    for lab, sc in zip(subset, subset_scores.tolist()):
        full_scores[name_to_idx[lab]] = float(sc)

    pred = [labels[i] for i in np.where(full_scores >= threshold)[0].tolist()]
    top_idx = np.argsort(-full_scores)[:topk].tolist()
    top_scored = [{"label": labels[i], "score": float(full_scores[i])} for i in top_idx]

    return {
        "list_prompt": list_res["prompt"],
        "list_raw_text": list_res["raw_text"],
        "list_json_ok": list_res.get("json_ok", None),
        "candidates": candidates,
        "subset_scored": subset,
        "yesno_prompt_preview_10": "\n\n".join(subset_prompt_preview),
        "threshold": threshold,
        "pred_labels": pred,
        "top_scored": top_scored,
        "scores": full_scores,
    }



In [ ]:
# Choose a few image ids (random or first K)
K = 3
chosen_img_ids = img_ids[:K]
chosen_img_ids


In [ ]:
threshold = 0.5
chunk_size = 4

for img_id in chosen_img_ids:
    info = coco.loadImgs(img_id)[0]
    path = os.path.join(IMG_DIR, info["file_name"])
    image = Image.open(path).convert("RGB")

    gt = get_gt_labels_for_img(img_id)

    print("\n" + "="*100)
    print("Image ID:", img_id, "| file:", info["file_name"])
    show_image(path, title=f"img_id={img_id}")

    # LIST
    list_out = run_list_mode(image, cat_names_sorted, max_new_tokens=128)
    print("\n[LIST MODE]")
    print("Pred labels:", list_out["pred_labels"])
    print("GT labels  :", gt)
    print("\nPrompt (first 600 chars):\n", list_out["prompt"][:600], "...\n")
    print("Raw output:\n", list_out["raw_text"][:800], "...\n")

    # YESNO
    yesno_out = run_yesno_mode(image, cat_names_sorted, threshold=threshold, chunk_size=chunk_size, topk=10)
    print("\n[YESNO MODE]")
    print("Pred labels:", yesno_out["pred_labels"])
    print("GT labels  :", gt)
    print("Top scored :", yesno_out["top_scored"])
    print("\nPrompt preview (first prompt):\n", yesno_out["prompt_preview_10"].split("\n\n")[0][:600], "...\n")

    # HYBRID
    hybrid_out = run_hybrid_mode(
        image, img_id, cat_names_sorted,
        threshold=threshold, chunk_size=chunk_size,
        max_new_tokens=128, num_neg=16, seed=123, max_candidates=20, topk=10
    )
    print("\n[HYBRID MODE]")
    print("Candidates:", hybrid_out["candidates"])
    print("Subset scored (len={}):".format(len(hybrid_out["subset_scored"])), hybrid_out["subset_scored"][:15], "...")
    print("Pred labels:", hybrid_out["pred_labels"])
    print("GT labels  :", gt)
    print("Top scored :", hybrid_out["top_scored"])
    print("Raw list output:\n", hybrid_out["list_raw_text"][:600], "...\n")
